In [1]:
pip install dash plotly

Note: you may need to restart the kernel to use updated packages.


In [2]:
!pip install dash-bootstrap-components

In [3]:
import pandas as pd
import plotly.express as px
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.cluster import KMeans
from dash import Dash, html, dcc
import dash_bootstrap_components as dbc
from jupyter_dash import JupyterDash  # Import JupyterDash instead of dash.Dash
from dash import dcc
from dash import html
import dash_bootstrap_components as dbc
from dash.dependencies import Input, Output
data = pd.read_pickle('OnlineRetail_data.pkl')

In [4]:
current_date = pd.to_datetime('2011-12-10')

rfm_data = data.groupby('CustomerID').agg({
    'InvoiceDate': lambda x: (current_date - x.max()).days,  
    'InvoiceNo': lambda x: len(x), 
    'TotalPrice': lambda x: x.sum()  
})

rfm_data.rename(columns={'InvoiceDate': 'Recency', 'InvoiceNo': 'Frequency', 'TotalPrice': 'Monetary'}, inplace=True)

kmeans = KMeans(n_clusters = 2, random_state = 42)  # Choose the number of clusters
rfm_data['Cluster'] = kmeans.fit_predict(rfm_data[['Recency', 'Frequency', 'Monetary']])

C:\Users\User\anaconda3\Lib\site-packages\sklearn\cluster\_kmeans.py:1446: UserWarning: KMeans is known to have a memory leak on Windows with MKL, when there are less chunks than available threads. You can avoid it by setting the environment variable OMP_NUM_THREADS=9.
  warnings.warn(


In [5]:
# Initialize the app
app = Dash(__name__, external_stylesheets=[dbc.themes.BOOTSTRAP])# Histogram for Recency

# Total Customers
total_customers = data['CustomerID'].nunique()

# Total Revenue
total_revenue = data['TotalPrice'].sum()

# Average Purchase Value
average_purchase_value = total_revenue / len(data)  # Per transaction or per purchase

#Average Purchase per Customer:
average_purchase_value_per_customer = total_revenue / total_customers

# Histogram for Recency
recency_histogram = px.histogram(rfm_data, x='Recency', nbins=30, 
                                  title='Recency Distribution',
                                  labels={'Recency': 'Recency (Days)'},
                                  color_discrete_sequence=['skyblue'])

# Histogram for Frequency
frequency_histogram = px.histogram(rfm_data, x='Frequency', nbins=30,
                                    title='Frequency Distribution',
                                    labels={'Frequency': 'Frequency (Number of Purchases)'},
                                    color_discrete_sequence=['lightgreen'])

# Histogram for Monetary
monetary_histogram = px.histogram(rfm_data, x='Monetary', nbins=30,
                                   title='Monetary Distribution',
                                   labels={'Monetary': 'Monetary Value (Total Amount Spent)'},
                                   color_discrete_sequence=['salmon'])

# Correlation heatmap
corr = rfm_data[['Recency', 'Frequency', 'Monetary']].corr()
rfm_heatmap = px.imshow(corr, 
                         title='Correlation Heatmap of RFM Metrics', 
                         color_continuous_scale='viridis',
                         text_auto=True,
                         labels=dict(x='Metrics', y='Metrics'))

# Customer segment concentration heatmap
heatmap_data = rfm_data.groupby('Cluster').mean().reset_index()
heatmap = px.imshow(
    heatmap_data[['Recency', 'Frequency', 'Monetary']].T,  # Transpose for correct orientation
    title='Heatmap of Average RFM Values by Customer Segment',
    color_continuous_scale='viridis',
    text_auto=True,
    labels=dict(x='Cluster', y='RFM Metrics'),
    x=heatmap_data['Cluster'])

# Assuming you have sales data by segment
revenue_data = rfm_data.groupby('Cluster')['Monetary'].sum().reset_index()
revenue = px.bar(revenue_data, x='Cluster', y='Monetary', title='Revenue Contribution by Customer Segment')

#Scatter for customer segment
scatter1 = px.scatter(rfm_data, x='Recency', y='Frequency', color='Cluster',
                 title='Customer Segments by RFM Values')

scatter2 = px.scatter(rfm_data, x='Recency', y='Monetary', color='Cluster',
                 title='Customer Segments by RFM Values')

# Pie chart for customer segmentation
segmentation_counts = rfm_data['Cluster'].value_counts()
pie = px.pie(values=segmentation_counts.values, names=segmentation_counts.index,
             title='Customer Segmentation Distribution')

cluster_monetary = rfm_data.groupby('Cluster')['Monetary'].sum().reset_index()
cluster_monetary.rename(columns={'Monetary': 'Total Monetary Value'}, inplace=True)
# Create a bar chart for Total Monetary Value by Cluster using Plotly
monetary_bar = px.bar(cluster_monetary,
                       x='Cluster',
                       y='Total Monetary Value',
                       color='Total Monetary Value',
                       title='Total Monetary Value per Customer Segment',
                       labels={'Total Monetary Value': 'Total Monetary Value ($)', 'Cluster': 'Customer Segment'},
                       color_continuous_scale='Viridis')

rfm_data['CLV'] = (rfm_data['Monetary'] * rfm_data['Frequency']) / rfm_data['Recency']
# Assuming you have CLV data by segment
clv_data = rfm_data.groupby('Cluster')['CLV'].mean().reset_index()
clv_bar = fig = px.bar(clv_data, x='Cluster', y='CLV', title='Average Customer Lifetime Value by Segment')

# Define churn criteria
churn_threshold = 265  
rfm_data['Churned'] = (rfm_data['Recency'] > churn_threshold).astype(int)
# Assuming you have customer churn data
churn_data = rfm_data.groupby('Churned').size().reset_index(name='Counts')
churn_bar = px.bar(churn_data, x='Churned', y='Counts', title='Customer Churn Rate')

# Now to calculate trends, we need to group by actual invoice dates
data['InvoiceDate'] = pd.to_datetime(data['InvoiceDate'])  # Ensure InvoiceDate is datetime
# Group by InvoiceDate and calculate the average RFM metrics
trend_data = data.groupby('InvoiceDate').agg({
    'InvoiceNo': 'count',  # Frequency
    'TotalPrice': 'sum'    # Monetary
}).reset_index()
# Calculate recency for daily trend, by counting days since last purchase
trend_data['Recency'] = (current_date - trend_data['InvoiceDate']).dt.days
# Create the line plot for the trend
trend_line = px.line(trend_data, x='InvoiceDate', y=['Recency', 'InvoiceNo', 'TotalPrice'],
              labels={'InvoiceNo': 'Frequency', 'TotalPrice': 'Monetary'},
              title='Trend Analysis of RFM Metrics Over Time')

# Define customer stages
new_customers = rfm_data[rfm_data['Frequency'] == 1]  
first_purchase = rfm_data[(rfm_data['Frequency'] == 1) & (rfm_data['Monetary'] > 0)] 
repeat_customers = rfm_data[rfm_data['Frequency'] > 1] 
loyal_customers = rfm_data[(rfm_data['Frequency'] > 3) & (rfm_data['Monetary'] > rfm_data['Monetary'].median())] 
# Count customers in each stage
funnel_data = {
    'Stage': ['New Customers', 'First Purchase', 'Repeat Customers', 'Loyal Customers'],
    'Count': [len(new_customers), len(first_purchase), len(repeat_customers), len(loyal_customers)]
}
# Create a DataFrame
funnel_df = pd.DataFrame(funnel_data)
# Use Plotly to create the funnel chart
sales_funnel = px.funnel(funnel_df, x='Count', y='Stage', title='Sales Funnel Visualization')

# Define the layout sections
app.layout = dbc.Container([
    # Header section
    dbc.Row(
        dbc.Col(html.H2("Customer Segmentation Dashboard", className="text-center mb-4"))
    ),

    # Sample KPIs for top section
    dbc.Row(
        [
            dbc.Col(dbc.Card([
                dbc.CardBody([
                    html.H5("Total Customers", className="card-title"),
                    html.H3(total_customers, className="card-text"),
                ], style={'height': '120px'})
            ], color="primary", inverse=True)),
            
            dbc.Col(dbc.Card([
                dbc.CardBody([
                    html.H5("Total Revenue", className="card-title"),
                    html.H3(f"{total_revenue:,.2f}", className="card-text"),
                ], style={'height': '120px'})
            ], color="info", inverse=True)),
            
            dbc.Col(dbc.Card([
                dbc.CardBody([
                    html.H5("Average Purchase Value", className="card-title"),
                    html.H3(f"{average_purchase_value:,.2f}", className="card-text"),
                ], style={'height': '120px'})
            ], color="success", inverse=True)),

            dbc.Col(dbc.Card([
                dbc.CardBody([
                    html.H5("Average Purchase Value Per Customer", className="card-title"),
                    html.H3(f"{average_purchase_value_per_customer:,.2f}", className="card-text"),
                ], style={'height': '120px'})
            ], color="secondary", inverse=True)),
        ],
        className="mb-4"
    ),
    
    # Section 1: RFM Distribution
    dbc.Row([
        dbc.Col(dcc.Graph(id='recency-histogram', figure=recency_histogram), width=6),
        dbc.Col(dcc.Graph(id='frequency-histogram', figure=frequency_histogram), width=6),
    ]),
    
    dbc.Row([
        dbc.Col(dcc.Graph(id='monetary-histogram', figure=monetary_histogram), width=6),
        dbc.Col(dcc.Graph(id='rfm-heatmap', figure=rfm_heatmap), width=6),
        dbc.Col(dcc.Graph(id='customer-heatmap', figure=heatmap), width=6),
        dbc.Col(dcc.Graph(id='revenue-bar', figure=revenue), width=6),
    ]),

    # Section 2: Customer Segmentation
    dbc.Row([
        dbc.Col(dcc.Graph(id='scatter-1', figure=scatter1), width=6),
        dbc.Col(dcc.Graph(id='scatter-2', figure=scatter2), width=6),
    ]),
    
    dbc.Row([
        dbc.Col(dcc.Graph(id='rfm-pie', figure=pie), width=6),
        dbc.Col(dcc.Graph(id='monetary-bar', figure=monetary_bar), width=6),
    ]),

    # Section 3: Churn & CLV Analysis
    dbc.Row([
        dbc.Col(dcc.Graph(id='churn-bar', figure=churn_bar), width=6),
        dbc.Col(dcc.Graph(id='clv-bar', figure=clv_bar), width=6),
    ]),

    # Section 4: Trend Analysis
    dbc.Row([
        dbc.Col(dcc.Graph(id='trend-line', figure=trend_line), width=12),
    ]),

    # Section 5: Sales Funnel
    dbc.Row([
        dbc.Col(dcc.Graph(id='sales-funnel', figure=sales_funnel), width=12),
    ]),
], fluid=True)

app.run_server(debug=True, port=8050)